# Model selection: which NVIDIA models can drive the agent and the judge

## Config

In [1]:
import json
import statistics
import sys
import urllib.request
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from agent.context import load_prompt, round_notice, starting_context
from agent.judge import judge
from agent.llm import LLM, LLMError, load_config
from agent.loop import ALL_TOOLS, INVESTIGATOR_PROMPT
from agent.schema import CRITERIA
from scripts import agent_smoke_test as smoke

pd.set_option("display.max_colwidth", 120)
config = load_config()
# every answer is cached under this tag, so a re-run reads the same answers and calls no model
CACHE_TAG = "model-selection"

## NVIDIA's live catalogue, and the candidates in it

In [2]:
with urllib.request.urlopen(config["base_url"] + "/models", timeout=30) as response:
    catalogue = sorted(model["id"] for model in json.load(response)["data"])
candidates = config["candidates"]
print(f"{len(catalogue)} models in the catalogue")
print(f"candidates not in it: {[model for model in candidates if model not in catalogue]}")
candidates

80 models in the catalogue
candidates not in it: []


['openai/gpt-oss-20b',
 'nvidia/nemotron-3.5-lightning-30b-a3b',
 'nvidia/nemotron-3-super-120b-a12b']

## Three checks of tool calling, on pretend tools

In [3]:
SYSTEM = {"role": "system", "content": "You help the operator of a compute cluster. Use the tools to answer. Never guess a fact a tool can give."}


def tool(name, description, argument):
    parameters = {"type": "object", "properties": {argument: {"type": "string"}}, "required": [argument]}
    return {"type": "function", "function": {"name": name, "description": description, "parameters": parameters}}


GET_STATUS = tool("get_node_status", "The current status of one node, for example node-3", "node")
GET_COMMAND = tool("get_last_command", "The last operator command that was run on one node", "node")
SUBMIT = tool("submit_answer", "Give the final answer", "answer")
# what the pretend tools return
TOOL_RESULTS = {"get_node_status": "node-12: not responding since 10:02:11", "get_last_command": "node-12: boot (command 4220) at 09:58:40"}


def first_call(reply):
    # the name and arguments of the first tool call of a reply; arguments are None when they are not valid JSON
    calls = reply.message.get("tool_calls") or []
    if not calls:
        return None, None
    function = calls[0]["function"]
    try:
        return function["name"], json.loads(function["arguments"])
    except (json.JSONDecodeError, TypeError):
        return function["name"], None


def check_tool_call(chat):
    # a question with one tool available: is the reply a well-formed call with the right argument?
    reply = chat([SYSTEM, {"role": "user", "content": "What is the status of node-12?"}], tools=[GET_STATUS])
    name, args = first_call(reply)
    return name == "get_node_status" and isinstance(args, dict) and args.get("node") == "node-12"


def check_two_step(chat):
    # call a tool, read its result, call a second tool, then answer from both results
    question = "What is the status of node-12, and what was the last operator command on it? Call one tool at a time."
    messages = [SYSTEM, {"role": "user", "content": question}]
    called = []
    for _ in range(4):
        reply = chat(messages, tools=[GET_STATUS, GET_COMMAND])
        messages.append(reply.message)
        calls = reply.message.get("tool_calls") or []
        if not calls:
            answer = reply.message.get("content") or ""
            return called[:2] == ["get_node_status", "get_last_command"] and "boot" in answer.lower()
        for call in calls:
            name = call["function"]["name"]
            called.append(name)
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": TOOL_RESULTS.get(name, "error: unknown tool")})
    return False


def check_forced_choice(chat):
    # is a forced tool choice obeyed?
    forced = {"type": "function", "function": {"name": "submit_answer"}}
    reply = chat([SYSTEM, {"role": "user", "content": "Say hello."}], tools=[GET_STATUS, SUBMIT], tool_choice=forced)
    name, args = first_call(reply)
    return name == "submit_answer" and isinstance(args, dict)


CHECKS = {"tool_call": check_tool_call, "two_step": check_two_step, "forced_choice": check_forced_choice}

## Run the checks on every candidate

In [4]:
def try_model(model):
    llm = LLM(config={**config, "models": {"candidate": model}})
    replies = []

    def chat(messages, **options):
        reply = llm.chat("candidate", messages, cache_tag=CACHE_TAG, **options)
        replies.append(reply)
        return reply

    row = {"model": model}
    for name, check in CHECKS.items():
        try:
            row[name] = "pass" if check(chat) else "fail"
        except LLMError as error:
            row[name] = f"error: {str(error)[:80]}"
    row["calls"] = len(replies)
    row["median_latency_s"] = round(statistics.median(r.latency_ms for r in replies) / 1000, 1) if replies else None
    row["tokens_in"] = sum(r.tokens_in for r in replies)
    row["tokens_out"] = sum(r.tokens_out for r in replies)
    return row


tool_calling = pd.DataFrame([try_model(model) for model in candidates])
tool_calling

,model,tool_call,two_step,forced_choice,calls,median_latency_s,tokens_in,tokens_out
0,openai/gpt-oss-20b,pass,pass,fail,5,1.0,1115,266
1,nvidia/nemotron-3.5-lightning-30b-a3b,pass,pass,pass,5,1.0,2026,355
2,nvidia/nemotron-3-super-120b-a12b,pass,pass,pass,5,0.8,2026,429


## The same candidates as judge: one call each on the plainly bad trajectory of the agent smoke test

In [5]:
tools, log = smoke.build_store()
bad_trajectory = smoke.bad_trajectory_text(tools, log)


def try_judge(model):
    # one attempt, no retry, at most a minute, and no extra request options
    llm = LLM(config={**config, "models": {"judge": model}, "options": {}, "max_retries": 0, "timeout_seconds": 60})
    try:
        judgement, replies = judge(llm, bad_trajectory, cache_tag=CACHE_TAG, retries=0)
    except (LLMError, ValueError) as error:
        return {"model": model, "answer": f"error: {str(error)[:80]}"}
    failed = [criterion for criterion in CRITERIA if getattr(judgement, criterion).result == "fail"]
    reply = replies[0]
    return {"model": model, "answer": "valid", "criteria_failed": f"{len(failed)} of {len(CRITERIA)}", "latency_s": round(reply.latency_ms / 1000, 1),
            "tokens_in": reply.tokens_in, "tokens_out": reply.tokens_out}


as_judge = pd.DataFrame([try_judge(model) for model in candidates])
as_judge

,model,answer,criteria_failed,latency_s,tokens_in,tokens_out
0,openai/gpt-oss-20b,valid,6 of 6,13.8,2663.0,997.0
1,nvidia/nemotron-3.5-lightning-30b-a3b,error: the call to nvidia/nemotron-3.5-lightning-30b-a3b failed: Request timed out.,NaN,NaN,NaN,NaN
2,nvidia/nemotron-3-super-120b-a12b,valid,5 of 6,30.1,2753.0,2997.0


## Hidden thinking on, off, and with a budget: the first round of the investigator on the smoke test's log, five times each

In [6]:
first_round = [
    {"role": "system", "content": load_prompt(INVESTIGATOR_PROMPT)},
    {"role": "user", "content": f"{starting_context(tools, log).text}\n\n{round_notice(1)}"},
]


def template(**kwargs):
    return {"extra_body": {"chat_template_kwargs": kwargs}}


# a budget is a cap, in tokens, on the thinking the model does before it answers
THINKING = {
    "on": {},
    "budget 500": template(enable_thinking=True, reasoning_budget=500),
    "budget 200": template(enable_thinking=True, reasoning_budget=200),
    "off": template(enable_thinking=False),
}


def try_thinking(setting, repeat):
    llm = LLM(config={**config, "options": {"main": THINKING[setting]}, "max_retries": 0, "timeout_seconds": 60})
    try:
        reply = llm.chat("main", first_round, tools=ALL_TOOLS, cache_tag=f"{CACHE_TAG}-{repeat}")
    except LLMError as error:
        return {"thinking": setting, "repeat": repeat, "tools_called": f"error: {str(error)[:80]}"}
    called = [call["function"]["name"] for call in reply.message.get("tool_calls") or []]
    return {"thinking": setting, "repeat": repeat, "latency_s": round(reply.latency_ms / 1000, 1), "tokens_in": reply.tokens_in,
            "tokens_out": reply.tokens_out, "n_tools_called": len(called), "tools_called": ", ".join(called)}


thinking = pd.DataFrame([try_thinking(setting, repeat) for setting in THINKING for repeat in range(1, 6)])
display(thinking)
thinking.groupby("thinking", sort=False)[["latency_s", "tokens_out", "n_tools_called"]].median()

,thinking,repeat,latency_s,tokens_in,tokens_out,n_tools_called,tools_called
0,on,1,18.6,5697,1924,4,"node_timeline, log_shapes_between, log_shapes_between, log_shapes_between"
1,on,2,10.1,5697,1141,5,"node_timeline, search_logs, search_logs, search_logs, log_shapes_between"
2,on,3,25.6,5697,1741,3,"node_timeline, search_logs, log_shapes_between"
3,on,4,18.0,5697,1834,2,"node_timeline, log_shapes_between"
4,on,5,15.6,5697,1534,3,"node_timeline, log_shapes_between, count_logs"
5,budget 500,1,13.9,5697,607,1,log_shapes_between
6,budget 500,2,7.3,5697,630,1,search_logs
7,budget 500,3,8.2,5697,631,1,search_logs
8,budget 500,4,8.0,5697,623,1,search_logs
9,budget 500,5,8.6,5697,641,1,node_timeline


,latency_s,tokens_out,n_tools_called
thinking,,,
on,18.0,1741.0,3.0
budget 500,8.2,630.0,1.0
budget 200,5.8,331.0,1.0
off,3.2,240.0,1.0


## The models in config/models.yaml

In [7]:
pd.Series(config["models"], name="model").to_frame()

,model
small,nvidia/nemotron-3.5-lightning-30b-a3b
main,nvidia/nemotron-3-super-120b-a12b
judge,openai/gpt-oss-20b
